# Module 1 Homework: Yellow Taxi Trip Records

This notebook solves the Module 1 homework assignment by analyzing Yellow Taxi Trip Records for January and February 2023. Uses sparse matrices to avoid MemoryError.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from scipy import sparse

In [ ]:
# Load January 2023 data from local file
df_jan = pd.read_parquet("yellow_tripdata_2023-01.parquet")

# Load February 2023 data from local file
df_feb = pd.read_parquet("yellow_tripdata_2023-02.parquet")

## Q1: Count number of columns in January data

In [ ]:
# Q1
n_columns = df_jan.shape[1]
print(f"Q1: Number of columns in January data: {n_columns}")

## Q2: Compute standard deviation of duration (in minutes)

In [ ]:
# Q2
# Calculate trip duration in minutes
df_jan['duration'] = (df_jan['tpep_dropoff_datetime'] - df_jan['tpep_pickup_datetime']).dt.total_seconds() / 60

duration_std = df_jan['duration'].std()
print(f"Q2: Standard deviation of duration (minutes): {duration_std:.4f}")

## Q3: Filter outliers (duration 1-60 min) and compute fraction remaining

In [ ]:
# Q3
n_original = len(df_jan)

# Filter outliers: keep only trips with duration between 1 and 60 minutes
df_jan_filtered = df_jan[(df_jan['duration'] >= 1) & (df_jan['duration'] <= 60)]

n_filtered = len(df_jan_filtered)
fraction_remaining = n_filtered / n_original

print(f"Q3: Fraction of trips remaining after filtering outliers: {fraction_remaining:.4f}")

## Q4: One-hot encode PULocationID and DOLocationID, report feature matrix dimensionality

In [ ]:
# Q4 - Use sparse matrices to avoid MemoryError
df = df_jan_filtered.copy()

pu_codes = df['PULocationID'].values.astype(np.int32)
do_codes = df['DOLocationID'].values.astype(np.int32)

# Offset DO codes so PU and DO locations are encoded separately
max_pu = int(df['PULocationID'].max())
max_do = int(df['DOLocationID'].max())

pu_indices = pu_codes
do_indices = do_codes + max_pu + 1

n_features = max_pu + max_do + 2

# Build sparse matrix with 2 non-zero entries per row
indices = np.arange(len(df))
rows = np.concatenate([indices, indices])
cols = np.concatenate([pu_indices, do_indices])
data = np.ones(len(rows), dtype=np.float64)

X_train = sparse.csr_matrix((data, (rows, cols)), shape=(len(df), n_features))

dimensionality = X_train.shape[1]
print(f"Q4: Feature matrix dimensionality after one-hot encoding: {dimensionality}")

## Q5: Train LinearRegression, report RMSE on training data

In [ ]:
# Q5
y_train = df['duration'].values

lr = LinearRegression()
lr.fit(X_train, y_train)

y_train_pred = lr.predict(X_train)

rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
print(f"Q5: RMSE on training data: {rmse_train:.4f}")

## Q6: Apply model to February data, report RMSE on validation

In [ ]:
# Q6
df_feb['duration'] = (df_feb['tpep_dropoff_datetime'] - df_feb['tpep_pickup_datetime']).dt.total_seconds() / 60
df_feb_filtered = df_feb[(df_feb['duration'] >= 1) & (df_feb['duration'] <= 60)]

feb_pu = df_feb_filtered['PULocationID'].values.astype(np.int32)
feb_do = df_feb_filtered['DOLocationID'].values.astype(np.int32)

# Apply same encoding as training data
feb_pu_indices = feb_pu
feb_do_indices = feb_do + max_pu + 1

n_val = len(df_feb_filtered)
val_indices = np.arange(n_val)

val_rows = np.concatenate([val_indices, val_indices])
val_cols = np.concatenate([feb_pu_indices, feb_do_indices])
val_data = np.ones(len(val_rows), dtype=np.float64)

X_val = sparse.csr_matrix((val_data, (val_rows, val_cols)), shape=(n_val, n_features))

y_val = df_feb_filtered['duration'].values

y_val_pred = lr.predict(X_val)

rmse_val = np.sqrt(mean_squared_error(y_val, y_val_pred))
print(f"Q6: RMSE on validation (February) data: {rmse_val:.4f}")